# Master Notebook: Pipeline Deteksi YOLOv8 + Dual VGG16 (Baju & Tas) + Analisis Warna K-Means + Streamlit
**Proyek Riset Sistem Pakar & Computer Vision — Prediksi Tren Fashion dari Media Sosial**  
Program Studi Informatika - Universitas Udayana

Notebook terpadu ini menggabungkan **kedua model klasifikasi (`baju` dan `tas`)** dalam satu alur kerja (*single unified notebook*) agar mudah dilatih, dievaluasi, dan diuji sekaligus:

### Alur Pipeline Terpadu:
1. **Tahap 1 — Setup & Konfigurasi Lingkungan:** Instalasi dan pemuatan pustaka (`ultralytics` YOLOv8, `torch`, `torchvision` VGG16, `scikit-learn` K-Means, `streamlit`, `Pillow`, `Matplotlib`).
2. **Tahap 2 — Deteksi & Auto-Cropping Dataset dengan YOLOv8 (Baju & Tas):**
   - Mendeteksi dan memotong (*crop*) bagian objek yang sesuai menggunakan **YOLOv8 (`yolov8n.pt`)** untuk kategori **Baju** (`casual_top`, `dress`, `outerwear`) dan **Tas** (`handbag`, `sling_bag`, `tote_bag`).
   - Menyesuaikan ukuran hasil potongan ke ukuran input standar VGG16 (**`224x224` piksel**) dan menyimpannya ke `dataset_cropped/baju` serta `dataset_cropped/tas`.
3. **Tahap 3 — Persiapan DataLoader Terstratifikasi (Bebas Error Multiprocessing & Class Mismatch):**
   - Pembagian data per kelas secara merata (**70% Train, 15% Validation, 15% Test**) dengan `num_workers=0` agar kompatibel 100% di Windows, Linux, macOS, dan Google Colab.
4. **Tahap 4 — Training & Evaluasi Kedua Model VGG16 (`best_vgg16_baju.pth` & `best_vgg16_tas.pth`):**
   - Melatih model VGG16 untuk **Baju** dan model VGG16 untuk **Tas** dari data hasil crop YOLOv8.
   - Menampilkan kurva *Loss & Accuracy*, *Confusion Matrix*, dan *Classification Report* (*Precision, Recall, F1-Score*) untuk masing-masing model.
5. **Tahap 5 — Uji Coba Kedua Model Sekaligus (YOLOv8 Crop $ightarrow$ Prediksi VGG16 $ightarrow$ Analisis Warna K-Means):**
   - Menguji foto baru dengan memotong bagian objek menggunakan YOLOv8 sesuai ukuran input model (`224x224`), memprediksi kelasnya menggunakan VGG16, lalu mengekstrak warna dominannya menggunakan **K-Means Clustering**.
6. **Tahap 6 — Pengujian Interaktif dengan Streamlit (`app.py`):**
   - Menyiapkan dan menjalankan dashboard aplikasi web **Streamlit** yang memuat kedua model (`best_vgg16_baju.pth` & `best_vgg16_tas.pth`) sekaligus.


## 1. Instalasi & Import Pustaka


In [ ]:
# Hapus tanda komentar pada baris di bawah jika pustaka belum terpasang:
# !pip install -q ultralytics opencv-python-headless scikit-learn tqdm seaborn matplotlib streamlit

import os
import time
import copy
import random
import zipfile
import urllib.request
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageDraw
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torchvision
from torchvision import transforms, datasets
from torchvision.models import vgg16, VGG16_Weights

from ultralytics import YOLO
from sklearn.cluster import KMeans
from sklearn.metrics import classification_report, confusion_matrix

# Konfigurasi Seed untuk Reprodusibilitas
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Hardware Acceleration : NVIDIA GPU ({torch.cuda.get_device_name(0)})")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Hardware Acceleration : Apple Silicon GPU (MPS)")
else:
    device = torch.device("cpu")
    print("Hardware Acceleration : CPU Mode")

INPUT_SIZE = (224, 224)
CATEGORIES = ["baju", "tas"]
print(f"PyTorch Version: {torch.__version__} | Target Input Size: {INPUT_SIZE}")

## 2. Akuisisi Dataset & Auto-Cropping YOLOv8 (Kategori Baju & Tas)
Seluruh gambar pada `dataset/baju` dan `dataset/tas` dideteksi menggunakan **YOLOv8 (`yolov8n.pt`)**, dipotong (*crop*) pada bagian objek yang sesuai, di-*resize* ke **`224x224`**, dan disimpan ke `dataset_cropped/baju` serta `dataset_cropped/tas`.


In [ ]:
RAW_DATASET_DIR = "dataset"
CROPPED_DATASET_DIR = "dataset_cropped"
ZIP_URL = "http://202.10.47.34/slides/dataset_fashion_labeled.zip"
LOCAL_ZIP = "dataset_fashion_labeled.zip"

if not (os.path.exists(os.path.join(RAW_DATASET_DIR, "baju")) and os.path.exists(os.path.join(RAW_DATASET_DIR, "tas"))):
    if not os.path.exists(LOCAL_ZIP):
        print(f"Mengunduh dataset dari {ZIP_URL}...")
        urllib.request.urlretrieve(ZIP_URL, LOCAL_ZIP)
        print("Unduhan selesai!")
    print("Mengekstrak berkas dataset...")
    with zipfile.ZipFile(LOCAL_ZIP, "r") as zf:
        zf.extractall(RAW_DATASET_DIR)
    print("Ekstraksi selesai!")

for cat in CATEGORIES:
    cat_dir = os.path.join(RAW_DATASET_DIR, cat)
    print(f"\nKategori '{cat.upper()}' ({cat_dir}):")
    for cls_name in sorted(os.listdir(cat_dir)):
        cls_path = os.path.join(cat_dir, cls_name)
        if os.path.isdir(cls_path):
            n_imgs = len([f for f in os.listdir(cls_path) if f.lower().endswith((".jpg", ".jpeg", ".png"))])
            print(f"  - {cls_name}: {n_imgs} gambar")

In [ ]:
# Muat model YOLOv8 untuk deteksi & cropping objek fesyen
yolo_model = YOLO("yolov8n.pt")
BAG_COCO_CLASSES = [24, 26, 28]  # backpack, handbag, suitcase
PERSON_COCO_CLASS = 0            # person

def detect_and_crop_yolov8(image_pil, category_mode="baju", target_size=(224, 224), conf_thres=0.20):
    """
    Mendeteksi bagian baju atau tas menggunakan YOLOv8, memotong (crop) area objek,
    dan mengubah ukurannya menjadi target_size (224x224) sesuai input VGG16.
    """
    img_rgb = np.array(image_pil.convert("RGB"))
    h, w, _ = img_rgb.shape

    results = yolo_model.predict(source=img_rgb, conf=conf_thres, verbose=False)
    boxes = results[0].boxes

    selected_box = None
    det_label = "center_fallback"
    det_conf = 1.0

    if boxes is not None and len(boxes) > 0:
        xyxy = boxes.xyxy.cpu().numpy()
        confs = boxes.conf.cpu().numpy()
        clss = boxes.cls.cpu().numpy().astype(int)

        if category_mode == "tas":
            bag_indices = [i for i, c in enumerate(clss) if c in BAG_COCO_CLASSES]
            if len(bag_indices) > 0:
                best_idx = max(bag_indices, key=lambda i: confs[i] * ((xyxy[i][2]-xyxy[i][0])*(xyxy[i][3]-xyxy[i][1])))
                selected_box = tuple(xyxy[best_idx])
                det_label = f"yolo_{yolo_model.names[clss[best_idx]]}"
                det_conf = float(confs[best_idx])
            else:
                person_indices = [i for i, c in enumerate(clss) if c == PERSON_COCO_CLASS]
                if len(person_indices) > 0:
                    best_idx = max(person_indices, key=lambda i: (xyxy[i][2]-xyxy[i][0])*(xyxy[i][3]-xyxy[i][1]))
                    px1, py1, px2, py2 = xyxy[best_idx]
                    pw, ph = (px2 - px1), (py2 - py1)
                    selected_box = (max(0, px1 - 0.05*pw), max(0, py1 + 0.18*ph), min(w, px2 + 0.05*pw), min(h, py1 + 0.90*ph))
                    det_label = "yolo_person_bag_region"
                    det_conf = float(confs[best_idx])
        else:  # category_mode == "baju"
            person_indices = [i for i, c in enumerate(clss) if c == PERSON_COCO_CLASS]
            if len(person_indices) > 0:
                best_idx = max(person_indices, key=lambda i: (xyxy[i][2]-xyxy[i][0])*(xyxy[i][3]-xyxy[i][1]))
                px1, py1, px2, py2 = xyxy[best_idx]
                pw, ph = (px2 - px1), (py2 - py1)
                selected_box = (max(0, px1 - 0.03*pw), max(0, py1 + 0.12*ph), min(w, px2 + 0.03*pw), min(h, py1 + 0.92*ph))
                det_label = "yolo_person_apparel"
                det_conf = float(confs[best_idx])
            else:
                best_idx = int(np.argmax(confs))
                selected_box = tuple(xyxy[best_idx])
                det_label = f"yolo_{yolo_model.names[clss[best_idx]]}"
                det_conf = float(confs[best_idx])

    if selected_box is None:
        mx, my = int(0.075 * w), int(0.075 * h)
        selected_box = (mx, my, w - mx, h - my)
        det_label = "center_object_crop"
        det_conf = 1.0

    x1, y1, x2, y2 = selected_box
    bw, bh = (x2 - x1), (y2 - y1)
    x1 = int(max(0, x1 - 0.04 * bw))
    y1 = int(max(0, y1 - 0.04 * bh))
    x2 = int(min(w, x2 + 0.04 * bw))
    y2 = int(min(h, y2 + 0.04 * bh))

    if (x2 - x1) < 32 or (y2 - y1) < 32:
        x1, y1, x2, y2 = 0, 0, w, h

    cropped_pil = image_pil.crop((x1, y1, x2, y2)).convert("RGB")
    cropped_resized_pil = cropped_pil.resize(target_size, Image.Resampling.LANCZOS)

    detection_info = {
        "method": det_label,
        "confidence": det_conf,
        "bbox": (x1, y1, x2, y2),
        "orig_size": (w, h),
        "crop_size": target_size
    }
    return cropped_resized_pil, (x1, y1, x2, y2), detection_info

In [ ]:
# Jalankan Auto-Cropping YOLOv8 untuk seluruh data BAJU dan TAS
for cat in CATEGORIES:
    raw_cat_dir = os.path.join(RAW_DATASET_DIR, cat)
    crop_cat_dir = os.path.join(CROPPED_DATASET_DIR, cat)
    os.makedirs(crop_cat_dir, exist_ok=True)

    cls_list = sorted([d for d in os.listdir(raw_cat_dir) if os.path.isdir(os.path.join(raw_cat_dir, d))])
    total_processed, yolo_hits = 0, 0

    print(f"\nMemproses Cropping YOLOv8 untuk Kategori '{cat.upper()}'...")
    for cls_name in cls_list:
        src_cls_dir = os.path.join(raw_cat_dir, cls_name)
        dst_cls_dir = os.path.join(crop_cat_dir, cls_name)
        os.makedirs(dst_cls_dir, exist_ok=True)

        img_files = sorted([f for f in os.listdir(src_cls_dir) if f.lower().endswith((".jpg", ".jpeg", ".png"))])
        for fname in tqdm(img_files, desc=f"[{cat.upper()}] {cls_name}", leave=False):
            src_path = os.path.join(src_cls_dir, fname)
            dst_path = os.path.join(dst_cls_dir, fname)

            with Image.open(src_path) as img:
                cropped_img, _, info = detect_and_crop_yolov8(img, category_mode=cat, target_size=INPUT_SIZE)
                cropped_img.save(dst_path, quality=95)
                total_processed += 1
                if info["method"].startswith("yolo_"):
                    yolo_hits += 1

    print(f"Selesai '{cat.upper()}': {total_processed} gambar disimpan ke '{crop_cat_dir}' (Deteksi BBox YOLOv8: {yolo_hits}/{total_processed})")

In [ ]:
# Visualisasi Sampel Sebelum & Sesudah Cropping YOLOv8 (Baju & Tas)
fig, axes = plt.subplots(2, 2, figsize=(10, 8))

for row_idx, cat in enumerate(CATEGORIES):
    raw_cat_dir = os.path.join(RAW_DATASET_DIR, cat)
    first_cls = sorted([d for d in os.listdir(raw_cat_dir) if os.path.isdir(os.path.join(raw_cat_dir, d))])[0]
    first_img = sorted(os.listdir(os.path.join(raw_cat_dir, first_cls)))[0]
    img_path = os.path.join(raw_cat_dir, first_cls, first_img)

    orig_pil = Image.open(img_path).convert("RGB")
    crop_pil, (x1, y1, x2, y2), info = detect_and_crop_yolov8(orig_pil, category_mode=cat, target_size=INPUT_SIZE)

    boxed = orig_pil.copy()
    draw = ImageDraw.Draw(boxed)
    draw.rectangle([x1, y1, x2, y2], outline="#ef4444", width=5)

    axes[row_idx, 0].imshow(boxed)
    axes[row_idx, 0].set_title(f"[{cat.upper()}] Citra Asli + BBox YOLOv8\n({info['method']}, conf={info['confidence']:.2f})", fontweight="bold")
    axes[row_idx, 0].axis("off")

    axes[row_idx, 1].imshow(crop_pil)
    axes[row_idx, 1].set_title(f"[{cat.upper()}] Hasil Crop YOLOv8 (224x224)\nKelas: {first_cls}", fontweight="bold")
    axes[row_idx, 1].axis("off")

plt.tight_layout()
plt.show()

## 3. DataLoader Terstratifikasi & Arsitektur Transfer Learning VGG16
Fungsi di bawah ini membagi dataset hasil crop YOLOv8 secara **terstratifikasi per kelas** (70% Train, 15% Validation, 15% Test) dan menggunakan `num_workers=0` untuk memastikan seluruh kelas terwakili secara seimbang serta bebas dari kendala *multiprocessing pickle* pada Jupyter Notebook.


In [ ]:
train_transforms = transforms.Compose([
    transforms.Resize(INPUT_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=12),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_test_transforms = transforms.Compose([
    transforms.Resize(INPUT_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class CroppedFashionDataset(Dataset):
    def __init__(self, sample_list, transform=None):
        self.samples = sample_list
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        img = Image.open(img_path).convert("RGB")
        if self.transform is not None:
            img = self.transform(img)
        return img, label

def create_stratified_dataloaders(cropped_cat_dir, batch_size=16):
    """Membagi data hasil crop YOLOv8 secara proporsional per kelas (70% Train, 15% Val, 15% Test)."""
    base_ds = datasets.ImageFolder(root=cropped_cat_dir)
    class_names = base_ds.classes

    # Kelompokkan sampel berdasarkan indeks kelas agar pembagian dijamin merata di setiap kelas
    per_class_samples = {i: [] for i in range(len(class_names))}
    for path, label in base_ds.samples:
        per_class_samples[label].append((path, label))

    train_samples, val_samples, test_samples = [], [], []
    rng = random.Random(SEED)

    for label, items in per_class_samples.items():
        items_copy = list(items)
        rng.shuffle(items_copy)
        n_total = len(items_copy)
        n_train = int(0.70 * n_total)
        n_val = max(1, int(0.15 * n_total))
        train_samples.extend(items_copy[:n_train])
        val_samples.extend(items_copy[n_train:n_train + n_val])
        test_samples.extend(items_copy[n_train + n_val:])

    rng.shuffle(train_samples)

    train_ds = CroppedFashionDataset(train_samples, transform=train_transforms)
    val_ds   = CroppedFashionDataset(val_samples, transform=val_test_transforms)
    test_ds  = CroppedFashionDataset(test_samples, transform=val_test_transforms)

    # num_workers=0 menjamin kompatibilitas penuh di Jupyter Notebook (Windows/Linux/macOS/Colab)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0)
    val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=0)
    test_loader  = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=0)

    return train_loader, val_loader, test_loader, class_names, (len(train_ds), len(val_ds), len(test_ds))

def build_vgg16_classifier(num_classes):
    model = vgg16(weights=VGG16_Weights.DEFAULT)
    for param in model.features.parameters():
        param.requires_grad = False
    for param in model.features[24:].parameters():
        param.requires_grad = True

    model.classifier = nn.Sequential(
        nn.Linear(512 * 7 * 7, 1024),
        nn.BatchNorm1d(1024),
        nn.ReLU(inplace=True),
        nn.Dropout(p=0.5),
        nn.Linear(1024, 512),
        nn.BatchNorm1d(512),
        nn.ReLU(inplace=True),
        nn.Dropout(p=0.4),
        nn.Linear(512, num_classes)
    )
    return model

## 4. Training & Evaluasi Kedua Model VGG16 (`baju` & `tas`)
Fungsi `train_and_evaluate_category()` di bawah ini melatih model VGG16 pada data hasil crop YOLOv8, menyimpan model terbaik ke `best_vgg16_baju.pth` dan `best_vgg16_tas.pth`, serta langsung menampilkan grafik kurva pelatihan, *Confusion Matrix*, dan *Classification Report*.


In [ ]:
def train_and_evaluate_category(category_name, num_epochs=15, batch_size=16):
    print("=" * 75)
    print(f"MEMULAI TRAINING & EVALUASI VGG16 UNTUK KATEGORI: {category_name.upper()}")
    print("=" * 75)

    cropped_cat_dir = os.path.join(CROPPED_DATASET_DIR, category_name)
    train_loader, val_loader, test_loader, class_names, split_counts = create_stratified_dataloaders(
        cropped_cat_dir, batch_size=batch_size
    )
    n_train, n_val, n_test = split_counts
    print(f"Kelas ({len(class_names)}) : {class_names}")
    print(f"Split Data  : Train={n_train} | Val={n_val} | Test={n_test}\n")

    model = build_vgg16_classifier(len(class_names)).to(device)
    criterion = nn.CrossEntropyLoss()
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = optim.AdamW(trainable_params, lr=1e-4, weight_decay=1e-3)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)

    save_path = f"best_vgg16_{category_name}.pth"
    best_wts = copy.deepcopy(model.state_dict())
    best_val_acc = 0.0
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

    since = time.time()
    for epoch in range(num_epochs):
        # --- Fase Training ---
        model.train()
        run_loss, run_corrects = 0.0, 0
        for inputs, labels in tqdm(train_loader, desc=f"[{category_name.upper()}] Epoch {epoch+1}/{num_epochs} (Train)", leave=False):
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            run_loss += loss.item() * inputs.size(0)
            run_corrects += torch.sum(preds == labels.data).item()

        epoch_train_loss = run_loss / max(1, n_train)
        epoch_train_acc = run_corrects / max(1, n_train)

        # --- Fase Validasi ---
        model.eval()
        val_loss, val_corrects = 0.0, 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * inputs.size(0)
                val_corrects += torch.sum(preds == labels.data).item()

        epoch_val_loss = val_loss / max(1, n_val)
        epoch_val_acc = val_corrects / max(1, n_val)
        scheduler.step(epoch_val_loss)

        history["train_loss"].append(epoch_train_loss)
        history["train_acc"].append(epoch_train_acc)
        history["val_loss"].append(epoch_val_loss)
        history["val_acc"].append(epoch_val_acc)

        saved_mark = ""
        if epoch_val_acc >= best_val_acc:
            best_val_acc = epoch_val_acc
            best_wts = copy.deepcopy(model.state_dict())
            torch.save({
                "model_state_dict": best_wts,
                "class_names": class_names,
                "category": category_name,
                "input_size": INPUT_SIZE
            }, save_path)
            saved_mark = f" [Checkpoint Disimpan -> {save_path}]"

        print(
            f"Epoch {epoch+1:02d}/{num_epochs:02d} | "
            f"Train Loss: {epoch_train_loss:.4f} ({epoch_train_acc*100:.1f}%) | "
            f"Val Loss: {epoch_val_loss:.4f} ({epoch_val_acc*100:.1f}%){saved_mark}"
        )

    elapsed = time.time() - since
    print(f"\nTraining {category_name.upper()} selesai dalam {elapsed//60:.0f}m {elapsed%60:.0f}s | Best Val Acc: {best_val_acc*100:.2f}%")
    model.load_state_dict(best_wts)

    # --- Visualisasi Kurva Loss & Akurasi ---
    epochs_range = range(1, len(history["train_loss"]) + 1)
    plt.figure(figsize=(14, 4.8))

    plt.subplot(1, 2, 1)
    plt.plot(epochs_range, history["train_loss"], "-o", label="Training Loss", color="#0284c7")
    plt.plot(epochs_range, history["val_loss"], "-s", label="Validation Loss", color="#ef4444")
    plt.title(f"Loss vs Epochs ({category_name.upper()})", fontsize=12, fontweight="bold")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(epochs_range, [a * 100 for a in history["train_acc"]], "-o", label="Training Accuracy", color="#0284c7")
    plt.plot(epochs_range, [a * 100 for a in history["val_acc"]], "-s", label="Validation Accuracy", color="#10b981")
    plt.title(f"Accuracy vs Epochs ({category_name.upper()})", fontsize=12, fontweight="bold")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy (%)")
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.legend()

    plt.tight_layout()
    plt.savefig(f"training_curves_{category_name}.png", dpi=300)
    plt.show()

    # --- Evaluasi pada Test Set (Bebas Error Mismatch Kelas) ---
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy().tolist())
            all_labels.extend(labels.numpy().tolist())

    label_indices = list(range(len(class_names)))
    print("\n" + "=" * 65)
    print(f"LAPORAN KLASIFIKASI MODEL VGG16 PADA TEST SET ({category_name.upper()})")
    print("=" * 65)
    print(classification_report(
        all_labels, all_preds,
        labels=label_indices,
        target_names=class_names,
        digits=4,
        zero_division=0
    ))

    cm = confusion_matrix(all_labels, all_preds, labels=label_indices)
    plt.figure(figsize=(6.5, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
    plt.title(f"Confusion Matrix — VGG16 ({category_name.upper()})", fontsize=12, fontweight="bold")
    plt.xlabel("Prediksi Model", fontweight="bold")
    plt.ylabel("Label Aktual (Ground Truth)", fontweight="bold")
    plt.tight_layout()
    plt.savefig(f"confusion_matrix_{category_name}.png", dpi=300)
    plt.show()

    return model, class_names, history

In [ ]:
# 4A. Jalankan Training & Evaluasi untuk Model 1: KATEGORI BAJU (casual_top, dress, outerwear)
EPOCHS = 15
model_baju, classes_baju, history_baju = train_and_evaluate_category("baju", num_epochs=EPOCHS, batch_size=16)

In [ ]:
# 4B. Jalankan Training & Evaluasi untuk Model 2: KATEGORI TAS (handbag, sling_bag, tote_bag)
model_tas, classes_tas, history_tas = train_and_evaluate_category("tas", num_epochs=EPOCHS, batch_size=16)

## 5. Uji Coba Kedua Model Sekaligus (YOLOv8 Crop $ightarrow$ Prediksi VGG16 Baju & Tas $ightarrow$ Analisis Warna K-Means)
Pada tahap ini, foto uji diproses melalui alur lengkap:
1. **Deteksi & Crop YOLOv8:** Memotong bagian objek sesuai ukuran input model (`224x224`).
2. **Prediksi VGG16:** Memprediksi kelas item menggunakan model yang telah dilatih (`best_vgg16_baju.pth` dan/atau `best_vgg16_tas.pth`).
3. **Analisis Warna K-Means:** Mengekstrak warna dominan beserta kode HEX, nama warna fesyen, dan persentase proporsinya.


In [ ]:
FASHION_COLOR_PALETTE = {
    "Hitam (Black)": (25, 25, 25),
    "Putih (White)": (245, 245, 245),
    "Abu-Abu Tua (Charcoal)": (75, 75, 80),
    "Abu-Abu Muda (Light Grey)": (185, 185, 190),
    "Krem / Beige": (225, 208, 185),
    "Putih Gading (Ivory)": (248, 242, 228),
    "Cokelat Muda (Tan / Camel)": (193, 154, 107),
    "Cokelat Tua (Dark Brown)": (92, 64, 51),
    "Cokelat Kopi (Espresso)": (58, 40, 32),
    "Biru Dongker (Navy Blue)": (28, 46, 74),
    "Biru Denim (Denim Blue)": (70, 105, 145),
    "Biru Muda (Sky / Baby Blue)": (155, 195, 225),
    "Merah Marun (Burgundy)": (115, 28, 38),
    "Merah Menyala (Crimson Red)": (195, 35, 45),
    "Merah Muda (Dusty Pink)": (215, 155, 165),
    "Merah Muda Cerah (Blush Pink)": (242, 195, 205),
    "Hijau Sage (Sage Green)": (158, 175, 152),
    "Hijau Olive (Olive Green)": (105, 115, 65),
    "Hijau Zamrud (Emerald)": (25, 110, 75),
    "Kuning Mustard": (215, 165, 45),
    "Oranye Terracotta": (195, 95, 65),
    "Ungu Lavender / Lilac": (185, 165, 205)
}

def get_closest_fashion_color_name(rgb_tuple):
    r, g, b = rgb_tuple
    return min(
        FASHION_COLOR_PALETTE.keys(),
        key=lambda name: (r - FASHION_COLOR_PALETTE[name][0])**2 +
                         (g - FASHION_COLOR_PALETTE[name][1])**2 +
                         (b - FASHION_COLOR_PALETTE[name][2])**2
    )

def analyze_dominant_colors_kmeans(cropped_pil, n_clusters=3):
    img_np = np.array(cropped_pil.convert("RGB"))
    h, w, _ = img_np.shape
    my, mx = int(0.125 * h), int(0.125 * w)
    core_region = img_np[my:h - my, mx:w - mx]
    pixels = core_region.reshape(-1, 3).astype(np.float32)

    kmeans = KMeans(n_clusters=n_clusters, random_state=SEED, n_init=10)
    labels = kmeans.fit_predict(pixels)
    centroids = kmeans.cluster_centers_.astype(int)
    counts = np.bincount(labels)
    total_pixels = len(labels)

    color_results = []
    for idx in np.argsort(-counts):
        rgb = tuple(int(v) for v in np.clip(centroids[idx], 0, 255))
        hex_code = "#{:02X}{:02X}{:02X}".format(*rgb)
        percentage = (counts[idx] / total_pixels) * 100.0
        color_results.append({
            "rgb": rgb,
            "hex": hex_code,
            "percentage": percentage,
            "name": get_closest_fashion_color_name(rgb)
        })
    return color_results

def load_trained_vgg16(category_name):
    ckpt_path = f"best_vgg16_{category_name}.pth"
    default_cls = ["casual_top", "dress", "outerwear"] if category_name == "baju" else ["handbag", "sling_bag", "tote_bag"]
    if os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location=device)
        cls_names = ckpt.get("class_names", default_cls)
        model = build_vgg16_classifier(len(cls_names)).to(device)
        model.load_state_dict(ckpt["model_state_dict"])
        model.eval()
        return model, cls_names
    else:
        fallback_model = model_baju if category_name == "baju" else model_tas
        fallback_cls = classes_baju if category_name == "baju" else classes_tas
        fallback_model.eval()
        return fallback_model, fallback_cls

def test_fashion_pipeline(image_source, categories=("baju", "tas"), n_colors=3):
    """
    Menguji gambar dengan satu atau kedua model sekaligus (Baju & Tas):
    YOLOv8 Crop (224x224) -> Prediksi VGG16 -> Analisis Warna K-Means.
    """
    if isinstance(categories, str):
        categories = [categories]

    if str(image_source).startswith(("http://", "https://")):
        req = urllib.request.Request(image_source, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req) as resp:
            orig_pil = Image.open(resp).convert("RGB")
    else:
        orig_pil = Image.open(image_source).convert("RGB")

    results_summary = {}

    for cat in categories:
        cropped_224_pil, (x1, y1, x2, y2), det_info = detect_and_crop_yolov8(
            orig_pil, category_mode=cat, target_size=INPUT_SIZE
        )
        infer_model, infer_classes = load_trained_vgg16(cat)

        input_tensor = val_test_transforms(cropped_224_pil).unsqueeze(0).to(device)
        with torch.no_grad():
            logits = infer_model(input_tensor)
            probs = torch.softmax(logits, dim=1)[0].cpu().numpy()

        top_idx = int(np.argmax(probs))
        pred_class = infer_classes[top_idx]
        pred_conf = float(probs[top_idx]) * 100.0
        dominant_colors = analyze_dominant_colors_kmeans(cropped_224_pil, n_clusters=n_colors)

        fig, axes = plt.subplots(1, 4, figsize=(18, 4.6))

        annotated_img = orig_pil.copy()
        draw = ImageDraw.Draw(annotated_img)
        draw.rectangle([x1, y1, x2, y2], outline="#ef4444", width=5)
        axes[0].imshow(annotated_img)
        axes[0].set_title(f"[{cat.upper()}] 1. Deteksi YOLOv8\n({det_info['method']}, conf={det_info['confidence']:.2f})", fontsize=10.5, fontweight="bold")
        axes[0].axis("off")

        axes[1].imshow(cropped_224_pil)
        axes[1].set_title(f"2. Hasil Crop ({INPUT_SIZE[0]}x{INPUT_SIZE[1]})\nPrediksi: {pred_class.upper()} ({pred_conf:.1f}%)", fontsize=10.5, fontweight="bold")
        axes[1].axis("off")

        y_pos = np.arange(len(infer_classes))
        bars = axes[2].barh(y_pos, probs * 100.0, color="#0284c7")
        axes[2].set_yticks(y_pos)
        axes[2].set_yticklabels(infer_classes, fontweight="bold")
        axes[2].set_xlim(0, 108)
        axes[2].set_xlabel("Confidence (%)")
        axes[2].set_title(f"3. Probabilitas VGG16 ({cat.upper()})", fontsize=10.5, fontweight="bold")
        axes[2].grid(axis="x", linestyle="--", alpha=0.5)
        for bar, p in zip(bars, probs * 100.0):
            axes[2].text(min(p + 2, 95), bar.get_y() + bar.get_height()/2, f"{p:.1f}%", va="center", fontsize=9, fontweight="bold")

        axes[3].set_title(f"4. Warna Dominan K-Means (Top {n_colors})", fontsize=10.5, fontweight="bold")
        y_cursor = 0.82
        for idx_c, c_info in enumerate(dominant_colors):
            rgb_norm = tuple(v / 255.0 for v in c_info["rgb"])
            rect = plt.Rectangle((0.04, y_cursor - 0.12), 0.20, 0.18, facecolor=rgb_norm, edgecolor="#334155", linewidth=1.5, transform=axes[3].transAxes)
            axes[3].add_patch(rect)
            axes[3].text(
                0.28, y_cursor - 0.03,
                f"#{idx_c+1}: {c_info['name']}\nHEX: {c_info['hex']} ({c_info['percentage']:.1f}%)",
                transform=axes[3].transAxes, fontsize=9.5, fontweight="bold", va="center"
            )
            y_cursor -= 0.28
        axes[3].axis("off")

        plt.tight_layout()
        plt.show()

        results_summary[cat] = {
            "predicted_class": pred_class,
            "confidence": pred_conf,
            "bbox": det_info["bbox"],
            "dominant_colors": dominant_colors
        }

    return results_summary

In [ ]:
# Uji Coba Otomatis Kedua Model (Model Baju & Model Tas) pada Sampel Dataset
sample_baju_dir = os.path.join(RAW_DATASET_DIR, "baju", classes_baju[0])
sample_baju_path = os.path.join(sample_baju_dir, sorted(os.listdir(sample_baju_dir))[0])
print(">>> UJI COBA SAMPEL BAJU (Menjalankan deteksi YOLOv8 -> VGG16 Baju -> K-Means):")
_ = test_fashion_pipeline(sample_baju_path, categories=["baju"], n_colors=3)

sample_tas_dir = os.path.join(RAW_DATASET_DIR, "tas", classes_tas[0])
sample_tas_path = os.path.join(sample_tas_dir, sorted(os.listdir(sample_tas_dir))[0])
print(">>> UJI COBA SAMPEL TAS (Menjalankan deteksi YOLOv8 -> VGG16 Tas -> K-Means):")
_ = test_fashion_pipeline(sample_tas_path, categories=["tas"], n_colors=3)

In [ ]:
# Uji Coba 1 Foto Sekaligus dengan KEDUA MODEL (Model Baju + Model Tas Bersamaan)
print(">>> UJI COBA DUAL-MODEL (1 Foto dianalisis oleh Model Baju & Model Tas sekaligus):")
_ = test_fashion_pipeline(sample_baju_path, categories=["baju", "tas"], n_colors=3)

## 6. Pengujian Interaktif Kedua Model Menggunakan Streamlit (`app.py`)
Cell di bawah ini memastikan berkas aplikasi **`app.py`** siap digunakan untuk menguji kedua model (`best_vgg16_baju.pth` dan `best_vgg16_tas.pth`) secara interaktif melalui antarmuka web **Streamlit**.

Cara menjalankan aplikasi Streamlit:
- **Di Komputer Lokal / Terminal VS Code:** Jalankan perintah `streamlit run app.py` di terminal, atau jalankan cell kedua di bawah ini.
- **Di Google Colab:** Gunakan *localtunnel* / *ngrok* atau jalankan cell perintah yang disediakan.


In [ ]:
# Menulis / memperbarui file aplikasi Streamlit (app.py) agar selalu sinkron dengan kedua model
streamlit_code = 'import os\nimport cv2\nimport numpy as np\nimport streamlit as st\nfrom PIL import Image, ImageDraw\nimport torch\nimport torch.nn as nn\nfrom torchvision import transforms\nfrom torchvision.models import vgg16, VGG16_Weights\nfrom ultralytics import YOLO\nfrom sklearn.cluster import KMeans\n\n# =====================================================================\n# KONFIGURASI HALAMAN STREAMLIT\n# =====================================================================\nst.set_page_config(\n    page_title="Fashion Trend AI — YOLOv8 + VGG16 + K-Means",\n    layout="wide"\n)\n\nst.title("Fashion Trend Analyzer: YOLOv8 + Dual VGG16 (Baju & Tas) + K-Means Color")\nst.markdown(\n    "Aplikasi pengujian terpadu untuk mendeteksi objek fesyen (**YOLOv8**), "\n    "mengklasifikasikan jenis **Baju** & **Tas** (**VGG16 Transfer Learning**), "\n    "serta mengekstrak palet warna dominan (**K-Means Clustering**)."\n)\n\nINPUT_SIZE = (224, 224)\nSEED = 42\n\nDEFAULT_CLASSES = {\n    "baju": ["casual_top", "dress", "outerwear"],\n    "tas": ["handbag", "sling_bag", "tote_bag"]\n}\n\nMODEL_PATHS = {\n    "baju": "best_vgg16_baju.pth",\n    "tas": "best_vgg16_tas.pth"\n}\n\nFASHION_COLOR_PALETTE = {\n    "Hitam (Black)": (25, 25, 25),\n    "Putih (White)": (245, 245, 245),\n    "Abu-Abu Tua (Charcoal)": (75, 75, 80),\n    "Abu-Abu Muda (Light Grey)": (185, 185, 190),\n    "Krem / Beige": (225, 208, 185),\n    "Putih Gading (Ivory)": (248, 242, 228),\n    "Cokelat Muda (Tan / Camel)": (193, 154, 107),\n    "Cokelat Tua (Dark Brown)": (92, 64, 51),\n    "Cokelat Kopi (Espresso)": (58, 40, 32),\n    "Biru Dongker (Navy Blue)": (28, 46, 74),\n    "Biru Denim (Denim Blue)": (70, 105, 145),\n    "Biru Muda (Sky / Baby Blue)": (155, 195, 225),\n    "Merah Marun (Burgundy)": (115, 28, 38),\n    "Merah Menyala (Crimson Red)": (195, 35, 45),\n    "Merah Muda (Dusty Pink)": (215, 155, 165),\n    "Merah Muda Cerah (Blush Pink)": (242, 195, 205),\n    "Hijau Sage (Sage Green)": (158, 175, 152),\n    "Hijau Olive (Olive Green)": (105, 115, 65),\n    "Hijau Zamrud (Emerald)": (25, 110, 75),\n    "Kuning Mustard": (215, 165, 45),\n    "Oranye Terracotta": (195, 95, 65),\n    "Ungu Lavender / Lilac": (185, 165, 205)\n}\n\nval_test_transforms = transforms.Compose([\n    transforms.Resize(INPUT_SIZE),\n    transforms.ToTensor(),\n    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])\n])\n\n@st.cache_resource\ndef get_device():\n    if torch.cuda.is_available():\n        return torch.device("cuda")\n    elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():\n        return torch.device("mps")\n    return torch.device("cpu")\n\n@st.cache_resource\ndef load_yolo():\n    return YOLO("yolov8n.pt")\n\ndef build_vgg16_classifier(num_classes):\n    model = vgg16(weights=VGG16_Weights.DEFAULT)\n    model.classifier = nn.Sequential(\n        nn.Linear(512 * 7 * 7, 1024),\n        nn.BatchNorm1d(1024),\n        nn.ReLU(inplace=True),\n        nn.Dropout(p=0.5),\n        nn.Linear(1024, 512),\n        nn.BatchNorm1d(512),\n        nn.ReLU(inplace=True),\n        nn.Dropout(p=0.4),\n        nn.Linear(512, num_classes)\n    )\n    return model\n\n@st.cache_resource\ndef load_vgg_model(category):\n    device = get_device()\n    ckpt_path = MODEL_PATHS[category]\n    classes = DEFAULT_CLASSES[category]\n    model = build_vgg16_classifier(len(classes)).to(device)\n    is_trained = False\n    if os.path.exists(ckpt_path):\n        ckpt = torch.load(ckpt_path, map_location=device)\n        if isinstance(ckpt, dict) and "model_state_dict" in ckpt:\n            classes = ckpt.get("class_names", classes)\n            model = build_vgg16_classifier(len(classes)).to(device)\n            model.load_state_dict(ckpt["model_state_dict"])\n        else:\n            model.load_state_dict(ckpt)\n        is_trained = True\n    model.eval()\n    return model, classes, is_trained\n\ndef detect_and_crop_yolov8(image_pil, yolo_model, category_mode="baju", target_size=(224, 224), conf_thres=0.20):\n    img_rgb = np.array(image_pil.convert("RGB"))\n    h, w, _ = img_rgb.shape\n    results = yolo_model.predict(source=img_rgb, conf=conf_thres, verbose=False)\n    boxes = results[0].boxes\n\n    BAG_COCO_CLASSES = [24, 26, 28]\n    PERSON_COCO_CLASS = 0\n    selected_box = None\n    det_label = "center_fallback"\n    det_conf = 1.0\n\n    if boxes is not None and len(boxes) > 0:\n        xyxy = boxes.xyxy.cpu().numpy()\n        confs = boxes.conf.cpu().numpy()\n        clss = boxes.cls.cpu().numpy().astype(int)\n\n        if category_mode == "tas":\n            bag_indices = [i for i, c in enumerate(clss) if c in BAG_COCO_CLASSES]\n            if len(bag_indices) > 0:\n                best_idx = max(bag_indices, key=lambda i: confs[i] * ((xyxy[i][2]-xyxy[i][0])*(xyxy[i][3]-xyxy[i][1])))\n                selected_box = tuple(xyxy[best_idx])\n                det_label = f"yolo_{yolo_model.names[clss[best_idx]]}"\n                det_conf = float(confs[best_idx])\n            else:\n                person_indices = [i for i, c in enumerate(clss) if c == PERSON_COCO_CLASS]\n                if len(person_indices) > 0:\n                    best_idx = max(person_indices, key=lambda i: (xyxy[i][2]-xyxy[i][0])*(xyxy[i][3]-xyxy[i][1]))\n                    px1, py1, px2, py2 = xyxy[best_idx]\n                    pw, ph = (px2 - px1), (py2 - py1)\n                    selected_box = (max(0, px1 - 0.05*pw), max(0, py1 + 0.18*ph), min(w, px2 + 0.05*pw), min(h, py1 + 0.90*ph))\n                    det_label = "yolo_person_bag_region"\n                    det_conf = float(confs[best_idx])\n        else:\n            person_indices = [i for i, c in enumerate(clss) if c == PERSON_COCO_CLASS]\n            if len(person_indices) > 0:\n                best_idx = max(person_indices, key=lambda i: (xyxy[i][2]-xyxy[i][0])*(xyxy[i][3]-xyxy[i][1]))\n                px1, py1, px2, py2 = xyxy[best_idx]\n                pw, ph = (px2 - px1), (py2 - py1)\n                selected_box = (max(0, px1 - 0.03*pw), max(0, py1 + 0.12*ph), min(w, px2 + 0.03*pw), min(h, py1 + 0.92*ph))\n                det_label = "yolo_person_apparel"\n                det_conf = float(confs[best_idx])\n            else:\n                best_idx = int(np.argmax(confs))\n                selected_box = tuple(xyxy[best_idx])\n                det_label = f"yolo_{yolo_model.names[clss[best_idx]]}"\n                det_conf = float(confs[best_idx])\n\n    if selected_box is None:\n        mx, my = int(0.075 * w), int(0.075 * h)\n        selected_box = (mx, my, w - mx, h - my)\n        det_label = "center_object_crop"\n        det_conf = 1.0\n\n    x1, y1, x2, y2 = selected_box\n    bw, bh = (x2 - x1), (y2 - y1)\n    x1 = int(max(0, x1 - 0.04 * bw))\n    y1 = int(max(0, y1 - 0.04 * bh))\n    x2 = int(min(w, x2 + 0.04 * bw))\n    y2 = int(min(h, y2 + 0.04 * bh))\n    if (x2 - x1) < 32 or (y2 - y1) < 32:\n        x1, y1, x2, y2 = 0, 0, w, h\n\n    cropped_pil = image_pil.crop((x1, y1, x2, y2)).convert("RGB")\n    cropped_resized_pil = cropped_pil.resize(target_size, Image.Resampling.LANCZOS)\n    return cropped_resized_pil, (x1, y1, x2, y2), {"method": det_label, "confidence": det_conf}\n\ndef get_closest_fashion_color_name(rgb_tuple):\n    r, g, b = rgb_tuple\n    return min(\n        FASHION_COLOR_PALETTE.keys(),\n        key=lambda name: (r - FASHION_COLOR_PALETTE[name][0])**2 +\n                         (g - FASHION_COLOR_PALETTE[name][1])**2 +\n                         (b - FASHION_COLOR_PALETTE[name][2])**2\n    )\n\ndef analyze_dominant_colors_kmeans(cropped_pil, n_clusters=3):\n    img_np = np.array(cropped_pil.convert("RGB"))\n    h, w, _ = img_np.shape\n    my, mx = int(0.125 * h), int(0.125 * w)\n    core_region = img_np[my:h - my, mx:w - mx]\n    pixels = core_region.reshape(-1, 3).astype(np.float32)\n\n    kmeans = KMeans(n_clusters=n_clusters, random_state=SEED, n_init=10)\n    labels = kmeans.fit_predict(pixels)\n    centroids = kmeans.cluster_centers_.astype(int)\n    counts = np.bincount(labels)\n    total_pixels = len(labels)\n\n    results = []\n    for idx in np.argsort(-counts):\n        rgb = tuple(int(v) for v in np.clip(centroids[idx], 0, 255))\n        hex_code = "#{:02X}{:02X}{:02X}".format(*rgb)\n        pct = (counts[idx] / total_pixels) * 100.0\n        results.append({\n            "rgb": rgb,\n            "hex": hex_code,\n            "percentage": pct,\n            "name": get_closest_fashion_color_name(rgb)\n        })\n    return results\n\n# =====================================================================\n# SIDEBAR & PENGATURAN PENGUJIAN\n# =====================================================================\nst.sidebar.header("Pengaturan Pipeline")\ntest_mode = st.sidebar.radio(\n    "Pilih Mode Pengujian Model:",\n    options=[\n        "Kedua Model Sekaligus (Baju & Tas)",\n        "Hanya Model Baju (VGG16 Baju)",\n        "Hanya Model Tas (VGG16 Tas)"\n    ],\n    index=0\n)\nn_clusters_color = st.sidebar.slider("Jumlah Cluster Warna (K-Means):", min_value=2, max_value=5, value=3)\n\nst.sidebar.markdown("---")\nst.sidebar.subheader("Status Checkpoint Model")\nfor cat_key, path_val in MODEL_PATHS.items():\n    if os.path.exists(path_val):\n        st.sidebar.success(f"Model {cat_key.upper()}: Siap ({path_val})")\n    else:\n        st.sidebar.warning(f"Model {cat_key.upper()}: Belum dilatih ({path_val} belum ditemukan)")\n\nuploaded_file = st.file_uploader("Unggah Foto Fesyen (.jpg, .jpeg, .png)", type=["jpg", "jpeg", "png"])\n\nif uploaded_file is not None:\n    orig_image = Image.open(uploaded_file).convert("RGB")\n    yolo_detector = load_yolo()\n    device = get_device()\n\n    categories_to_run = ["baju", "tas"]\n    if "Hanya Model Baju" in test_mode:\n        categories_to_run = ["baju"]\n    elif "Hanya Model Tas" in test_mode:\n        categories_to_run = ["tas"]\n\n    for cat in categories_to_run:\n        st.markdown("---")\n        st.subheader(f"Hasil Analisis Kategori: {cat.upper()}")\n\n        cropped_224, (x1, y1, x2, y2), det_info = detect_and_crop_yolov8(\n            orig_image, yolo_detector, category_mode=cat, target_size=INPUT_SIZE\n        )\n        vgg_model, class_list, is_trained = load_vgg_model(cat)\n\n        input_tensor = val_test_transforms(cropped_224).unsqueeze(0).to(device)\n        with torch.no_grad():\n            logits = vgg_model(input_tensor)\n            probs = torch.softmax(logits, dim=1)[0].cpu().numpy()\n\n        top_idx = int(np.argmax(probs))\n        pred_label = class_list[top_idx]\n        pred_conf = float(probs[top_idx]) * 100.0\n        colors_info = analyze_dominant_colors_kmeans(cropped_224, n_clusters=n_clusters_color)\n\n        # Gambar Bounding Box pada Citra Asli\n        boxed_img = orig_image.copy()\n        draw = ImageDraw.Draw(boxed_img)\n        box_color = "#2563eb" if cat == "baju" else "#10b981"\n        draw.rectangle([x1, y1, x2, y2], outline=box_color, width=5)\n\n        col1, col2, col3, col4 = st.columns([1.2, 1.0, 1.3, 1.5])\n\n        with col1:\n            st.markdown("**1. Deteksi YOLOv8**")\n            st.image(boxed_img, use_container_width=True)\n            st.caption(f"Metode: `{det_info[\'method\']}` | Conf: `{det_info[\'confidence\']:.2f}`")\n\n        with col2:\n            st.markdown(f"**2. Hasil Crop ({INPUT_SIZE[0]}x{INPUT_SIZE[1]})**")\n            st.image(cropped_224, use_container_width=True)\n            st.caption(f"BBox: `{x1}, {y1}, {x2}, {y2}`")\n\n        with col3:\n            st.markdown(f"**3. Prediksi VGG16 ({cat.upper()})**")\n            st.metric(label="Kelas Terprediksi", value=pred_label.upper(), delta=f"{pred_conf:.2f}% Confidence")\n            for c_name, p_val in zip(class_list, probs * 100.0):\n                st.write(f"`{c_name}`: **{p_val:.1f}%**")\n                st.progress(min(max(int(p_val), 0), 100))\n\n        with col4:\n            st.markdown(f"**4. Warna Dominan (K-Means Top {n_clusters_color})**")\n            for idx_c, c_item in enumerate(colors_info, 1):\n                swatch_html = f"""\n                <div style="display:flex;align-items:center;gap:12px;margin-bottom:10px;padding:8px;border-radius:8px;background:#f8fafc;border:1px solid #e2e8f0;">\n                    <div style="width:42px;height:42px;border-radius:6px;background:{c_item[\'hex\']};border:1px solid #334155;flex-shrink:0;"></div>\n                    <div style="font-size:13px;line-height:1.4;color:#0f172a;">\n                        <strong>#{idx_c} {c_item[\'name\']}</strong><br>\n                        HEX: <code>{c_item[\'hex\']}</code> | Proporsi: <strong>{c_item[\'percentage\']:.1f}%</strong>\n                    </div>\n                </div>\n                """\n                st.markdown(swatch_html, unsafe_allow_html=True)\n'
with open("app.py", "w", encoding="utf-8") as f:
    f.write(streamlit_code)
print("File 'app.py' berhasil disiapkan! Siap dijalankan dengan perintah: streamlit run app.py")

In [ ]:
# Hapus tanda komentar pada baris di bawah ini untuk langsung menjalankan server Streamlit dari dalam Notebook:
# !streamlit run app.py --server.port 8501